# Setup Lakebase Autoscale

Creates the Lakebase Autoscale project and app-writable tables:
1. Create Autoscale project `db-residential-copilot` (idempotent)
2. Create `dbx_res_app` schema
3. Create `deal_scenarios` and `chat_audit` tables

**Note:** `portfolio_metrics` and `portfolio_time_series` are synced tables created in notebook 03.

In [ ]:
%pip install -U "databricks-sdk>=0.81.0" "psycopg[binary]>=3.0"
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

PROJECT_ID = "db-residential-copilot"
BRANCH = "production"
DATABASE = "databricks_postgres"

## Step 1: Create Lakebase Autoscale Project

In [ ]:
from databricks.sdk.service.postgres import (
    Project, ProjectSpec, Endpoint, EndpointSpec, EndpointType
)

# Check if project already exists
existing_projects = list(w.postgres.list_projects())
project_exists = any(
    p.name == f"projects/{PROJECT_ID}" for p in existing_projects
)

if project_exists:
    print(f"Project '{PROJECT_ID}' already exists — skipping creation.")
    project = w.postgres.get_project(name=f"projects/{PROJECT_ID}")
else:
    print(f"Creating Lakebase Autoscale project '{PROJECT_ID}'...")
    project = w.postgres.create_project(
        project=Project(
            spec=ProjectSpec(
                display_name="DB Residential Copilot",
                pg_version="17"
            )
        ),
        project_id=PROJECT_ID
    ).wait()
    print(f"Project created: {project.name}")

print(f"Project: {project.name}")

# Wait for endpoint to be available
import time
branch_name = f"projects/{PROJECT_ID}/branches/{BRANCH}"
for attempt in range(30):
    endpoints = list(w.postgres.list_endpoints(parent=branch_name))
    if endpoints:
        ep = endpoints[0]
        print(f"Endpoint ready: {ep.name}")
        break
    print(f"  Waiting for endpoint... (attempt {attempt + 1})")
    time.sleep(10)
else:
    raise RuntimeError("No endpoints found after waiting")

## Step 3: Create App Schema and Tables

Connect via psycopg to create the `dbx_res_app` schema and app-writable tables.

In [ ]:
import psycopg

# Get connection details
branch_name = f"projects/{PROJECT_ID}/branches/{BRANCH}"
endpoints = list(w.postgres.list_endpoints(parent=branch_name))
ep = w.postgres.get_endpoint(name=endpoints[0].name)
host = ep.status.hosts.host

# Generate credential
cred = w.postgres.generate_database_credential(endpoint=endpoints[0].name)
username = w.current_user.me().user_name

print(f"Connecting to {host} as {username}...")

conn = psycopg.connect(
    host=host,
    port=5432,
    dbname=DATABASE,
    user=username,
    password=cred.token,
    sslmode="require"
)
conn.autocommit = True
print("Connected.")

In [ ]:
cur = conn.cursor()

# Create app schema
cur.execute("CREATE SCHEMA IF NOT EXISTS dbx_res_app;")
print("Schema 'dbx_res_app' ready.")

# deal_scenarios: stores user-submitted deal underwriting scenarios
cur.execute("""
CREATE TABLE IF NOT EXISTS dbx_res_app.deal_scenarios (
    deal_id TEXT PRIMARY KEY DEFAULT gen_random_uuid()::text,
    property_name TEXT NOT NULL,
    city TEXT,
    state TEXT,
    property_type TEXT,
    purchase_price NUMERIC(14, 2) NOT NULL,
    units INTEGER NOT NULL DEFAULT 1,
    monthly_rent_per_unit NUMERIC(10, 2) NOT NULL,
    ltv_pct NUMERIC(5, 2) DEFAULT 75.0,
    interest_rate_pct NUMERIC(5, 3) DEFAULT 6.5,
    loan_term_years INTEGER DEFAULT 30,
    rent_growth_pct NUMERIC(5, 2) DEFAULT 3.0,
    expense_ratio_pct NUMERIC(5, 2) DEFAULT 35.0,
    exit_cap_rate_pct NUMERIC(5, 3) DEFAULT 5.5,
    hold_years INTEGER DEFAULT 5,
    -- Computed outputs (filled by forecast)
    noi NUMERIC(14, 2),
    dscr NUMERIC(6, 3),
    cash_on_cash_pct NUMERIC(6, 2),
    irr_pct NUMERIC(6, 2),
    equity_multiple NUMERIC(6, 3),
    npv NUMERIC(14, 2),
    created_at TIMESTAMPTZ DEFAULT now(),
    updated_at TIMESTAMPTZ DEFAULT now()
);
""")
print("Table 'dbx_res_app.deal_scenarios' ready.")

# chat_audit: logs copilot questions and answers
cur.execute("""
CREATE TABLE IF NOT EXISTS dbx_res_app.chat_audit (
    chat_id TEXT PRIMARY KEY DEFAULT gen_random_uuid()::text,
    user_email TEXT,
    question TEXT NOT NULL,
    answer_summary TEXT,
    model_endpoint TEXT,
    latency_ms INTEGER,
    created_at TIMESTAMPTZ DEFAULT now()
);
""")
print("Table 'dbx_res_app.chat_audit' ready.")

cur.close()
print("\nAll app tables created successfully.")

## Validate

In [ ]:
cur = conn.cursor()
cur.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'dbx_res_app'
ORDER BY table_name;
""")
print("Tables in 'dbx_res_app' schema:")
for row in cur.fetchall():
    print(f"  {row[0]}.{row[1]}")

cur.close()
conn.close()
print("\nLakebase setup complete.")